In [1]:
import sys

sys.path.append("../")

import sqlite3
import pandas as pd

pd.options.mode.chained_assignment = None

import os

from dotenv import load_dotenv

load_dotenv()

DB_SCIENCE_PATH_NEW  = os.getenv("DB_SCIENCE_PATH_NEW")

conn = sqlite3.connect(DB_SCIENCE_PATH_NEW)

In [21]:
df_occupation = pd.read_sql("SELECT * FROM cleaned_occupations_science", conn)
df_main_information = pd.read_sql("SELECT * FROM individuals_occupation_information", conn)
df_main_information = df_main_information.drop('productive_year', axis=1)

df_occupation = df_occupation.rename(columns={'wikidata_id':'individual_wikidata_id'})

df_final = pd.merge(df_occupation, df_main_information, on = 'individual_wikidata_id')
df_final = df_final[['individual_wikidata_id', 'meta_occupation', 'individual_name', 'birthyear']].copy()
df_final = df_final.groupby(['individual_wikidata_id', 'individual_name', 'birthyear'])['meta_occupation'].apply(lambda x : ' | '.join(x))
df_final = df_final.reset_index()

df_regions = pd.read_sql("SELECT * FROM individuals_regions", conn)
df_regions = df_regions[['individual_wikidata_id', 'region_name']].drop_duplicates()
df_regions = df_regions.groupby('individual_wikidata_id')['region_name'].apply(lambda x : ' | '.join(x))
df_regions = df_regions.reset_index()
df_final = pd.merge(df_final, df_regions, on = 'individual_wikidata_id')
df_final.to_csv('data/scientists.csv')